In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

# 1. Konfigurácia a parametre
SEED = 42
TRAIN_SHARE = 0.70
EMPLOYMENT_SENTINEL = 365243

BUILDING_COLUMNS = [
    "APARTMENTS_AVG",
    "COMMONAREA_AVG",
    "LIVINGAREA_AVG",
    "NONLIVINGAREA_AVG",
    "HOUSETYPE_MODE",
    "WALLSMATERIAL_MODE",
    "EMERGENCYSTATE_MODE",
]

# 2. Pomocné funkcie
def stratified_split(
    data: pd.DataFrame, target_col: str = "TARGET"
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Vytvorí deterministické 70/30 rozdelenie so zachovaním podielu cieľovej premennej."""
    if target_col not in data.columns:
        raise ValueError(f"Požadovaný stĺpec {target_col!r} chýba.")
    if data[target_col].isna().any():
        raise ValueError("TARGET obsahuje chýbajúce hodnoty.")

    labels = set(data[target_col].unique())
    if labels != {0, 1}:
        raise ValueError(f"Očakávané hodnoty TARGET sú 0 a 1; našli sa {labels}.")

    rng = np.random.default_rng(SEED)
    train_indices: list[int] = []
    validation_indices: list[int] = []

    for label in sorted(labels):
        group_indices = data.index[data[target_col] == label].to_numpy().copy()
        rng.shuffle(group_indices)
        n_train = int(TRAIN_SHARE * len(group_indices))
        train_indices.extend(group_indices[:n_train].tolist())
        validation_indices.extend(group_indices[n_train:].tolist())

    train_idx = np.asarray(train_indices, dtype=int)
    validation_idx = np.asarray(validation_indices, dtype=int)
    rng.shuffle(train_idx)
    rng.shuffle(validation_idx)

    train = data.loc[train_idx].copy()
    validation = data.loc[validation_idx].copy()
    return train, validation


def add_initial_features(data: pd.DataFrame) -> pd.DataFrame:
    """Pridá odvodené premenné."""
    data = data.copy()

    data["AGE_YEARS"] = -data["DAYS_BIRTH"] / 365.25

    employed_days = data["DAYS_EMPLOYED"]
    is_employment_sentinel = employed_days.eq(EMPLOYMENT_SENTINEL)
    data["EMPLOYMENT_SPECIAL_CODE"] = np.where(
        employed_days.isna(), np.nan, is_employment_sentinel.astype(float)
    )
    data["EMPLOYMENT_YEARS"] = (-employed_days / 365.25).where(
        ~is_employment_sentinel
    )

    for source, destination in [
        ("DAYS_REGISTRATION", "YEARS_SINCE_REGISTRATION"),
        ("DAYS_ID_PUBLISH", "YEARS_SINCE_ID_PUBLISH"),
        ("DAYS_LAST_PHONE_CHANGE", "YEARS_SINCE_PHONE_CHANGE"),
    ]:
        data[destination] = -data[source] / 365.25

    income = data["AMT_INCOME_TOTAL"].replace(0, np.nan)
    goods_price = data["AMT_GOODS_PRICE"].replace(0, np.nan)
    data["CREDIT_TO_INCOME"] = data["AMT_CREDIT"] / income
    data["ANNUITY_TO_INCOME"] = data["AMT_ANNUITY"] / income
    data["CREDIT_TO_GOODS_PRICE"] = data["AMT_CREDIT"] / goods_price

    available_building_columns = [
        column for column in BUILDING_COLUMNS if column in data.columns
    ]
    if len(available_building_columns) != len(BUILDING_COLUMNS):
        missing = sorted(set(BUILDING_COLUMNS) - set(available_building_columns))
        raise ValueError(f"Chýbajúce stĺpce pre budovy: {missing}")
    data["BUILDING_MISSING_COUNT"] = data[available_building_columns].isna().sum(axis=1)

    return data


def make_training_quality_summary(train: pd.DataFrame) -> pd.DataFrame:
    """Sumár kvality trénovacích dát."""
    rows = []
    n = len(train)

    for column in train.columns:
        series = train[column]
        nonmissing = series.dropna()
        record = {
            "variable": column,
            "dtype": str(series.dtype),
            "n_rows": n,
            "n_missing": int(series.isna().sum()),
            "missing_pct": float(series.isna().mean()),
            "n_unique_nonmissing": int(nonmissing.nunique()),
        }

        if pd.api.types.is_numeric_dtype(series) and not nonmissing.empty:
            quantiles = nonmissing.quantile([0.01, 0.50, 0.99])
            record.update(
                {
                    "minimum": float(nonmissing.min()),
                    "p01": float(quantiles.loc[0.01]),
                    "median": float(quantiles.loc[0.50]),
                    "p99": float(quantiles.loc[0.99]),
                    "maximum": float(nonmissing.max()),
                }
            )
        else:
            record.update(
                {"minimum": np.nan, "p01": np.nan, "median": np.nan,
                 "p99": np.nan, "maximum": np.nan}
            )
        rows.append(record)

    return pd.DataFrame(rows)


def make_split_summary(train: pd.DataFrame, validation: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for name, subset in [("train", train), ("validation", validation)]:
        rows.append(
            {
                "sample": name,
                "n_total": len(subset),
                "n_good_TARGET_0": int((subset["TARGET"] == 0).sum()),
                "n_bad_TARGET_1": int((subset["TARGET"] == 1).sum()),
                "bad_rate_in_sample": float(subset["TARGET"].mean()),
                "random_seed": SEED,
                "train_share": TRAIN_SHARE,
            }
        )
    return pd.DataFrame(rows)

# 3. Spustenie výpočtu nad AppData.xlsx
input_path = Path("AppData.xlsx")
output_dir = Path("task1_output")

if not input_path.exists():
    raise FileNotFoundError(f"Súbor {input_path} sa nenašiel v aktívnom priečinku.")

raw = pd.read_excel(input_path)

# Kontrola stĺpcov
required = {"SK_ID_CURR", "TARGET", "DAYS_BIRTH", "DAYS_EMPLOYED",
            "DAYS_REGISTRATION", "DAYS_ID_PUBLISH", "DAYS_LAST_PHONE_CHANGE",
            "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"}
missing_columns = sorted(required - set(raw.columns))
if missing_columns:
    raise ValueError(f"Chýbajú požadované stĺpce: {missing_columns}")

# Rozdelenie a spracovanie
train, validation = stratified_split(raw)
train = add_initial_features(train)
validation = add_initial_features(validation)

# Uloženie výstupov
output_dir.mkdir(parents=True, exist_ok=True)
train.to_csv(output_dir / "train.csv", index=False)
validation.to_csv(output_dir / "validation.csv", index=False)
make_split_summary(train, validation).to_csv(output_dir / "split_summary.csv", index=False)
make_training_quality_summary(train).to_csv(output_dir / "training_data_quality.csv", index=False)

# Prehľad
summary = make_split_summary(train, validation)
print(f"Seed: {SEED}; Stratified split: {TRAIN_SHARE:.0%}/{1-TRAIN_SHARE:.0%}\n")
print(summary.to_string(index=False))